# Demo: tracing temporal labels to saved predictions

This notebook is for the **presentation video**. It does **not** retrain a model. It loads the saved held-out Exp004 visual and Exp005 audio prediction CSVs that were used to calculate the reported metrics.

The point of the demo is to show that I can trace a sample from its condition and modality labels to its 64 temporal targets, model scores and interpretation.

## Recording sequence

1. Run the setup and loading cells.
2. Point out the columns that define the sample and temporal target.
3. Show **Case A**: both modalities fake, with a useful visual response around the manipulated positions.
4. Show **Case B**: audio-only fake, where the visual target is zero everywhere but visual scores still become very high.
5. Return to the slide deck.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

VISUAL_THRESHOLD = 0.245   # selected on Exp004 validation balanced accuracy
AUDIO_THRESHOLD = 0.250    # selected on Exp005 validation balanced accuracy

plt.rcParams["figure.figsize"] = (10.5, 6.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

def find_repo_root(start=None):
    """Walk upwards until a repository containing experiments/ is found."""
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "experiments").is_dir():
            return path
    raise FileNotFoundError("Could not find a repository root containing experiments/")

def first_existing(candidates, label):
    """Return the first existing path from a small list of canonical/historical candidates."""
    for path in candidates:
        if path.exists():
            return path
    joined = "\n  - ".join(str(p) for p in candidates)
    raise FileNotFoundError(f"Could not find {label}. Tried:\n  - {joined}")

REPO_ROOT = find_repo_root()
EXP_DIR = REPO_ROOT / "experiments"
print("Repository root:", REPO_ROOT)

In [ ]:
VISUAL_PRED_PATH = first_existing([
    EXP_DIR / "exp004_visual_temporal_bal40k" / "results" / "test_frame_predictions.csv",
    EXP_DIR / "exp004_visual_temporal_bal40k" / "test_frame_predictions.csv",
    EXP_DIR / "exp004_visual_temporal" / "results" / "test_frame_predictions.csv",
], "Exp004 visual test predictions")

AUDIO_PRED_PATH = first_existing([
    EXP_DIR / "exp005_audio_temporal_bal40k" / "results" / "test_window_predictions.csv",
    EXP_DIR / "exp005_audio_temporal_bal40k" / "test_window_predictions.csv",
    EXP_DIR / "exp005_audio_temporal" / "results" / "test_window_predictions.csv",
], "Exp005 audio test predictions")

visual = pd.read_csv(VISUAL_PRED_PATH)
audio = pd.read_csv(AUDIO_PRED_PATH)

required_visual = {"row_index", "condition", "visual_fake", "audio_fake", "binary_label", "timestep", "frame_true", "frame_prob"}
required_audio = {"row_index", "condition", "visual_fake", "audio_fake", "binary_label", "window_idx", "window_true", "window_prob"}
assert required_visual.issubset(visual.columns), sorted(required_visual - set(visual.columns))
assert required_audio.issubset(audio.columns), sorted(required_audio - set(audio.columns))

print("Visual file:", VISUAL_PRED_PATH)
print("Audio file:", AUDIO_PRED_PATH)
print("Visual:", f"{len(visual):,} rows / {visual['row_index'].nunique():,} clips")
print("Audio:", f"{len(audio):,} rows / {audio['row_index'].nunique():,} clips")
print("Rows per visual clip:", visual.groupby("row_index").size().unique())
print("Rows per audio clip:", audio.groupby("row_index").size().unique())

### What the important columns mean

- `condition`: real, video-only fake, audio-only fake, or both fake.
- `visual_fake` / `audio_fake`: modality-level clip targets.
- `frame_true` / `window_true`: temporal target at one of the 64 sampled positions.
- `frame_prob` / `window_prob`: model fake probability for that sampled position.

This is the evidence chain used in the demo: **condition → modality target → temporal target → model score → interpretation**.

In [ ]:
def clip_summary(df, true_col, prob_col, index_col="row_index"):
    """Collapse temporal rows to a compact per-clip summary for the demo."""
    return (
        df.groupby(index_col)
          .agg(
              condition=("condition", "first"),
              visual_fake=("visual_fake", "first"),
              audio_fake=("audio_fake", "first"),
              binary_label=("binary_label", "first"),
              true_positions=(true_col, "sum"),
              max_prob=(prob_col, "max"),
              mean_prob=(prob_col, "mean"),
          )
          .reset_index()
    )

visual_summary = clip_summary(visual, "frame_true", "frame_prob")
audio_summary = clip_summary(audio, "window_true", "window_prob")

print("Visual condition counts (clips):")
print(visual_summary["condition"].value_counts().to_string())
print("\nAudio condition counts (clips):")
print(audio_summary["condition"].value_counts().to_string())

In [ ]:
def plot_temporal_case(row_index, title=None):
    """Plot visual and audio temporal predictions against their modality-specific targets."""
    vv = visual[visual["row_index"] == row_index].sort_values("timestep")
    aa = audio[audio["row_index"] == row_index].sort_values("window_idx")
    if vv.empty or aa.empty:
        raise ValueError(f"Missing visual or audio predictions for row_index={row_index}")

    condition = vv["condition"].iloc[0]
    fig, axes = plt.subplots(2, 1, figsize=(10.5, 6.4), sharex=True)

    ax = axes[0]
    ax.plot(vv["timestep"], vv["frame_prob"], linewidth=2, label="visual fake probability")
    ax.axhline(VISUAL_THRESHOLD, linestyle="--", linewidth=1.2, label=f"visual threshold = {VISUAL_THRESHOLD:.3f}")
    ax.fill_between(vv["timestep"], 0, 1, where=vv["frame_true"].astype(bool), alpha=0.16, label="visual GT manipulated")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Probability")
    ax.set_title("Visual branch")
    ax.legend(loc="upper right", fontsize=8)

    ax = axes[1]
    ax.plot(aa["window_idx"], aa["window_prob"], linewidth=2, label="audio fake probability")
    ax.axhline(AUDIO_THRESHOLD, linestyle="--", linewidth=1.2, label=f"audio threshold = {AUDIO_THRESHOLD:.3f}")
    ax.fill_between(aa["window_idx"], 0, 1, where=aa["window_true"].astype(bool), alpha=0.16, label="audio GT manipulated")
    ax.set_ylim(0, 1)
    ax.set_xlabel("Sampled temporal position (0–63)")
    ax.set_ylabel("Probability")
    ax.set_title("Audio branch")
    ax.legend(loc="upper right", fontsize=8)

    fig.suptitle(title or f"row_index={row_index} | {condition.replace('_', ' ')}", y=1.01)
    plt.tight_layout()
    return fig, axes


def print_case_summary(row_index):
    """Print the clip-level labels and compact temporal score summaries."""
    vs = visual_summary[visual_summary["row_index"] == row_index]
    aus = audio_summary[audio_summary["row_index"] == row_index]
    if vs.empty or aus.empty:
        raise ValueError(f"No summary found for row_index={row_index}")

    print("Clip labels")
    print(vs[["row_index", "condition", "binary_label", "visual_fake", "audio_fake"]].to_string(index=False))
    print("\nVisual temporal summary")
    print(vs[["true_positions", "max_prob", "mean_prob"]].to_string(index=False))
    print("\nAudio temporal summary")
    print(aus[["true_positions", "max_prob", "mean_prob"]].to_string(index=False))

## Case A — fake video + fake audio

This is a useful positive example. The visual ground truth is positive only at a few sampled positions, and the visual probability peaks strongly around them. The broader activation outside the labelled interval also illustrates why good AUC can coexist with weak precision/IoU.

In [ ]:
CASE_A = 868
print_case_summary(CASE_A)
plot_temporal_case(CASE_A, "Case A — fake video + fake audio")
plt.show()

## Case B — real video + fake audio, but the visual branch fires

This is the central failure example. Because the video is genuine, `visual_fake = 0` and `frame_true = 0` at every sampled visual position. Despite that, the visual branch reaches a very high fake probability.

In [ ]:
CASE_B = 3515
print_case_summary(CASE_B)
plot_temporal_case(CASE_B, "Case B — audio-only fake, visual branch fires")
plt.show()

## Interpretation to say out loud

- **Case A:** the detector has useful local ranking signal, but the activation is broader than the narrow manipulation label.
- **Case B:** the visual branch is highly confident on an audio-only fake even though the visual target is zero everywhere.

Together, these show why the project distinguishes **binary correctness** from **modality-specific and temporal evidence**.

In [ ]:
# Optional fallback: find alternative cases if the saved row indices ever change.
print("Strong visually manipulated examples:")
display(
    visual_summary[(visual_summary["visual_fake"] == 1) & (visual_summary["true_positions"] > 0)]
    .sort_values("max_prob", ascending=False)
    .head(8)
)

print("\nAudio-only examples with high visual scores:")
display(
    visual_summary[visual_summary["condition"] == "real_video_fake_audio"]
    .sort_values("max_prob", ascending=False)
    .head(8)
)